In [19]:
# ---------------------- 修复NumPy导入问题 ----------------------
import numpy as np
import sys
import types

# 安全地创建必要的NumPy类型
if 'numpy._typing' not in sys.modules:
    sys.modules['numpy._typing'] = types.ModuleType('numpy._typing')

class _NBitLong: pass
class ArrayLike: pass
class DTypeLike: pass
class NBitBase: pass
class NDArray: pass

# 添加到numpy._typing模块
numpy_typing = sys.modules['numpy._typing']
setattr(numpy_typing, '_NBitLong', _NBitLong)
setattr(numpy_typing, 'ArrayLike', ArrayLike)
setattr(numpy_typing, 'DTypeLike', DTypeLike)
setattr(numpy_typing, 'NBitBase', NBitBase)
setattr(numpy_typing, 'NDArray', NDArray)

# 创建numpy.typing模块
if 'numpy.typing' not in sys.modules:
    sys.modules['numpy.typing'] = types.ModuleType('numpy.typing')
numpy_typing_public = sys.modules['numpy.typing']
setattr(numpy_typing_public, 'ArrayLike', ArrayLike)
setattr(numpy_typing_public, 'DTypeLike', DTypeLike)
setattr(numpy_typing_public, 'NBitBase', NBitBase)
setattr(numpy_typing_public, 'NDArray', NDArray)

# ---------------------- 导入必要的库 ----------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn import metrics
import pandas as pd
import argparse
import os
import sys

# ---------------------- 解决Jupyter参数问题 ----------------------
# 检查是否在Jupyter环境中运行
is_jupyter = 'ipykernel' in sys.modules

# 设置默认模式（在Jupyter中默认为训练模式）
if is_jupyter:
    sys.argv = [sys.argv[0], '--mode', 'train']

# ---------------------- 解析命令行参数 ----------------------
parser = argparse.ArgumentParser(description="BERT + CNN 情感分类器")
parser.add_argument('--mode', choices=['train', 'test'], required=True, help='运行模式: train or test')
parser.add_argument('--model_path', type=str, default='./model/cnn_bert_final.model', help='模型保存/加载路径')
parser.add_argument('--epochs', type=int, default=5, help='训练轮数')
parser.add_argument('--batch_size', type=int, default=24, help='批处理大小')
parser.add_argument('--lr', type=float, default=5e-4, help='学习率')
args = parser.parse_args([]) if is_jupyter and len(sys.argv) < 2 else parser.parse_args()

# ---------------------- 设置设备和超参 ----------------------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
num_filters = 128
filter_sizes = [2, 3, 4]
max_length = 128

# ---------------------- 加载BERT模型（使用本地路径） ----------------------
# 定义BERT模型本地路径 - 使用原始字符串和双反斜杠
bert_local_path = r'G:\中文情感识别\bert_base_uncased'

# 确保路径存在
if not os.path.exists(bert_local_path):
    raise FileNotFoundError(f"BERT本地路径不存在: {bert_local_path}")

print(f"从本地路径加载BERT模型: {bert_local_path}")

# 直接从transformers导入，避免在线下载
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained(bert_local_path)
bert_model = BertModel.from_pretrained(bert_local_path).to(device)

# ---------------------- 读取数据 ----------------------
def load_corpus_bert(path):
    """加载CSV格式的数据集"""
    try:
        df = pd.read_csv(path)
        
        # 检查列名 - 解决列名大小写问题
        column_mapping = {}
        for col in df.columns:
            col_lower = col.lower()
            if 'text' in col_lower and 'text' not in column_mapping:
                column_mapping['text'] = col
            elif 'label' in col_lower and 'label' not in column_mapping:
                column_mapping['label'] = col
        
        # 确保找到了必要的列
        if 'text' not in column_mapping or 'label' not in column_mapping:
            print(f"文件 {path} 中未找到 'text' 或 'label' 列。实际列名: {df.columns.tolist()}")
            return []
        
        # 使用找到的列名提取数据
        return [(row[column_mapping['text']], row[column_mapping['label']]) for _, row in df.iterrows()]
    except Exception as e:
        print(f"加载数据失败: {e}")
        return []

# 修复路径问题：使用原始字符串和双反斜杠
# 确保路径格式正确
BASE_DIR = r'G:\中文情感识别'
DATA_DIR = os.path.join(BASE_DIR, 'data', 'ChnSentiCorp_htl_all')
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
DEV_PATH = os.path.join(DATA_DIR, 'dev.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')

# 确保数据路径存在
print("检查数据路径...")
for path in [TRAIN_PATH, DEV_PATH, TEST_PATH]:
    if not os.path.exists(path):
        # 尝试不同编码方式打开文件
        for encoding in ['utf-8', 'gbk', 'latin1']:
            try:
                with open(path, 'r', encoding=encoding) as f:
                    pass
                print(f"文件存在但编码可能是 {encoding}: {path}")
                break
            except:
                continue
        raise FileNotFoundError(f"数据文件不存在: {path}")
    else:
        print(f"文件存在: {path}")

print(f"\n加载训练数据: {TRAIN_PATH}")
train_data = load_corpus_bert(TRAIN_PATH)
print(f"加载验证数据: {DEV_PATH}")
dev_data = load_corpus_bert(DEV_PATH)
print(f"加载测试数据: {TEST_PATH}")
test_data = load_corpus_bert(TEST_PATH)

# 检查数据是否加载成功
if len(train_data) == 0:
    # 尝试使用不同的列名
    print("尝试使用备用列名...")
    try:
        df_train = pd.read_csv(TRAIN_PATH)
        print(f"文件列名: {df_train.columns.tolist()}")
        
        # 尝试常见的列名变体
        text_col = None
        label_col = None
        
        possible_text_cols = ['text', 'Text', 'TEXT', 'content', 'review', 'sentence', 'comment']
        possible_label_cols = ['label', 'Label', 'LABEL', 'sentiment', 'score', 'rating', 'class']
        
        for col in df_train.columns:
            if col.lower() in possible_text_cols:
                text_col = col
            if col.lower() in possible_label_cols:
                label_col = col
                
        if text_col and label_col:
            print(f"使用备用列名: text={text_col}, label={label_col}")
            train_data = [(row[text_col], row[label_col]) for _, row in df_train.iterrows()]
    
    except Exception as e:
        print(f"备用加载失败: {e}")
        raise ValueError("训练数据为空，请检查数据文件格式")

if len(train_data) == 0:
    raise ValueError("训练数据为空，请检查数据文件格式")
if len(dev_data) == 0:
    raise ValueError("验证数据为空，请检查数据文件格式")
if len(test_data) == 0:
    raise ValueError("测试数据为空，请检查数据文件格式")

print(f"训练样本数: {len(train_data)}")
print(f"验证样本数: {len(dev_data)}")
print(f"测试样本数: {len(test_data)}")

# 创建DataFrame并打印示例
df_train = pd.DataFrame(train_data, columns=["text", "label"])
df_dev = pd.DataFrame(dev_data, columns=["text", "label"])
df_test = pd.DataFrame(test_data, columns=["text", "label"])

print("\n训练数据示例:")
print(df_train.head())
print("\n标签分布:")
print(df_train['label'].value_counts())

# ---------------------- 自定义 Dataset 数据类 ----------------------
class MyDataset(Dataset):
    def __init__(self, df):
        self.data = []
        self.label = df["label"].tolist()
        print(f"正在处理 {len(self.label)} 个样本...")
        
        for i, s in enumerate(df["text"].tolist()):
            try:
                # 确保文本是字符串
                if not isinstance(s, str):
                    s = str(s)
                    
                inputs = tokenizer(
                    s, 
                    padding='max_length', 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=max_length
                )
                self.data.append(inputs)
                
                # 显示进度
                if (i + 1) % 100 == 0:
                    print(f"已处理 {i+1}/{len(self.label)} 个样本")
            except Exception as e:
                print(f"处理文本失败: {e}")
                print(f"问题文本: {s[:50] if isinstance(s, str) else '非字符串类型'}...")

    def __getitem__(self, index):
        return self.data[index], float(self.label[index])

    def __len__(self):
        return len(self.label)

# ---------------------- 定义 collate_fn 合并数据批 ----------------------
def collate_fn(data):
    try:
        input_ids = [item[0]['input_ids'].squeeze(0) for item in data]
        attention_mask = [item[0]['attention_mask'].squeeze(0) for item in data]
        labels = [item[1] for item in data]
        
        input_ids = torch.stack(input_ids)
        attention_mask = torch.stack(attention_mask)
        return input_ids, attention_mask, torch.tensor(labels)
    except Exception as e:
        print(f"数据批处理失败: {e}")
        # 打印问题数据
        for i, item in enumerate(data):
            print(f"样本 {i}:")
            print(f"输入ID形状: {item[0]['input_ids'].shape}")
            print(f"注意力掩码形状: {item[0]['attention_mask'].shape}")
            print(f"标签: {item[1]}")
        raise

# ---------------------- 创建数据输入模块 ----------------------
print("\n创建训练数据加载器...")
train_loader = DataLoader(MyDataset(df_train), batch_size=args.batch_size, collate_fn=collate_fn, shuffle=True)
print("\n创建测试数据加载器...")
test_loader = DataLoader(MyDataset(df_test), batch_size=args.batch_size, collate_fn=collate_fn, shuffle=False)

# ---------------------- BERT + CNN 模型 ----------------------
class BERT_CNN(nn.Module):
    def __init__(self, filter_sizes, num_filters):
        super(BERT_CNN, self).__init__()
        self.bert = bert_model
        # 冻结BERT参数
        for param in self.bert.parameters():
            param.requires_grad = False

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, 768)) for k in filter_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state

        x = x.unsqueeze(1) 
        conv_outputs = []
        for conv in self.convs:
            conv_out = torch.relu(conv(x)).squeeze(3) 
            pooled = torch.max(conv_out, dim=2)[0]    
            conv_outputs.append(pooled)
            
        x = torch.cat(conv_outputs, dim=1) 
        x = self.dropout(x)
        x = self.fc(x)
        return self.sigmoid(x)

# ---------------------- 模型评价函数 ----------------------
def evaluate(model, data_loader):
    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for i, (input_ids, attention_mask, labels) in enumerate(data_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()
            outputs = model(input_ids, attention_mask).view(-1)
            y_pred.append(outputs.cpu())
            y_true.append(labels.cpu())
            
            # 显示进度
            if (i + 1) % 10 == 0:
                print(f"已评估 {i+1}/{len(data_loader)} 批次")

    y_prob = torch.cat(y_pred).numpy()
    y_true = torch.cat(y_true).numpy()
    y_label = (y_prob > 0.5).astype(int)

    print("\n分类报告:")
    print(metrics.classification_report(y_true, y_label))
    print(f"准确率: {metrics.accuracy_score(y_true, y_label):.4f}")
    print(f"AUC: {metrics.roc_auc_score(y_true, y_prob):.4f}")
    print(f"F1分数: {metrics.f1_score(y_true, y_label):.4f}")

# ---------------------- 训练函数 ----------------------
def train():
    model = BERT_CNN(filter_sizes, num_filters).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    
    # 创建模型目录
    os.makedirs(os.path.dirname(args.model_path), exist_ok=True)
    
    print(f"\n开始训练，共 {args.epochs} 个周期，批量大小: {args.batch_size}，学习率: {args.lr}")
    print(f"模型结构: {model}")
    
    for epoch in range(args.epochs):
        model.train()
        total_loss, total_samples = 0, 0
        print(f"\n{'='*50}")
        print(f"周期 {epoch + 1}/{args.epochs}")
        
        for i, (input_ids, attention_mask, labels) in enumerate(train_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()

            outputs = model(input_ids, attention_mask).view(-1)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            
            if (i + 1) % 10 == 0:
                avg_loss = total_loss / total_samples
                print(f"批次 {i + 1}/{len(train_loader)} - 损失: {avg_loss:.4f}")
                total_loss, total_samples = 0, 0

        # 每个epoch结束后在验证集上评估
        print(f"\n周期 {epoch+1}/{args.epochs} 验证结果:")
        evaluate(model, test_loader)

        # 保存模型
        torch.save(model.state_dict(), args.model_path)
        print(f"模型已保存至 {args.model_path}")

# ---------------------- 模型测试函数 ----------------------
def test():
    model = BERT_CNN(filter_sizes, num_filters).to(device)
    model.load_state_dict(torch.load(args.model_path, map_location=device))
    print("模型加载成功，测试结果:")
    evaluate(model, test_loader)

# ---------------------- 主程序运行入口 ----------------------
if __name__ == '__main__':
    print(f"\n{'='*50}")
    print(f"运行模式: {args.mode}")
    print(f"设备: {device}")
    print(f"模型路径: {args.model_path}")
    print(f"训练轮数: {args.epochs}")
    print(f"批量大小: {args.batch_size}")
    print(f"学习率: {args.lr}")
    print(f"卷积核大小: {filter_sizes}")
    print(f"卷积核数量: {num_filters}")
    print(f"最大长度: {max_length}")
    print(f"数据目录: {DATA_DIR}")
    print(f"{'='*50}\n")
    
    if args.mode == 'train':
        train()
    elif args.mode == 'test':
        test()
    else:
        print(f"未知模式: {args.mode}")

从本地路径加载BERT模型: G:\中文情感识别\bert_base_uncased
检查数据路径...
文件存在: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
文件存在: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
文件存在: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv

加载训练数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
文件 G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv 中未找到 'text' 或 'label' 列。实际列名: ['0', '服务态度极其差，前台接待好象没有受过培训，连基本的礼貌都不懂，竟然同时接待几个客人；大堂副理更差，跟客人辩解个没完，要总经理的电话投诉竟然都不敢给。要是没有作什么亏心事情，跟本不用这么怕。']
加载验证数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
文件 G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv 中未找到 'text' 或 'label' 列。实际列名: ['1', '环境还可以，房间比较干净，就是有点交通不便，其他都挺好的']
加载测试数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv
文件 G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv 中未找到 'text' 或 'label' 列。实际列名: ['1', '距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.']
尝试使用备用列名...
文件列名: ['0', '服务态度极其差，前台接待好象没有受过培训，连基本的礼貌都不懂，竟然同时接待几个客人；大堂副理更差，跟客人辩解个没完，要总经理的电话投诉竟然都不敢给。要是没有作什么亏心事情，跟本不用这么怕。']


ValueError: 训练数据为空，请检查数据文件格式

In [21]:
import pandas as pd
from datasets import Dataset

def load_fixed_data(file_path):
    """处理特殊格式的酒店评论数据"""
    # 手动读取原始文本
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    data = []
    current_label = None
    current_text = []
    
    # 特殊格式解析
    for line in lines:
        # 标签行检测 (以数字开头)
        if line.startswith(('0', '1')) and ',' in line:
            # 保存前一条记录
            if current_label is not None:
                data.append({'label': int(current_label), 'text': "".join(current_text)})
            
            # 开始新记录
            parts = line.strip().split(',', 1)  # 只分割第一个逗号
            current_label = parts[0]
            current_text = [parts[1] if len(parts) > 1 else ""]
        else:
            # 延续文本内容
            current_text.append(line)
    
    # 添加最后一条记录
    if current_label is not None:
        data.append({'label': int(current_label), 'text': "".join(current_text)})
    
    return Dataset.from_pandas(pd.DataFrame(data))

# 加载数据集
train_data = load_fixed_data("G:\\中文情感识别\\data\\ChnSentiCorp_htl_all\\train.csv")
dev_data = load_fixed_data("G:\\中文情感识别\\data\\ChnSentiCorp_htl_all\\dev.csv")
test_data = load_fixed_data("G:\\中文情感识别\\data\\ChnSentiCorp_htl_all\\test.csv")

In [22]:
print(f"训练集样本: {len(train_data)}")
print(f"首条样本: {train_data[0]}")

训练集样本: 4982
首条样本: {'label': 0, 'text': '服务态度极其差，前台接待好象没有受过培训，连基本的礼貌都不懂，竟然同时接待几个客人；大堂副理更差，跟客人辩解个没完，要总经理的电话投诉竟然都不敢给。要是没有作什么亏心事情，跟本不用这么怕。'}


In [24]:
import numpy as np
import sys
import types
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn import metrics
import pandas as pd
import argparse
import os
import time
from transformers import BertTokenizer, BertModel

# ====================== 解决NumPy导入问题 ======================
if 'numpy._typing' not in sys.modules:
    sys.modules['numpy._typing'] = types.ModuleType('numpy._typing')

class _NBitLong: pass
class ArrayLike: pass
class DTypeLike: pass
class NBitBase: pass
class NDArray: pass

numpy_typing = sys.modules['numpy._typing']
setattr(numpy_typing, '_NBitLong', _NBitLong)
setattr(numpy_typing, 'ArrayLike', ArrayLike)
setattr(numpy_typing, 'DTypeLike', DTypeLike)
setattr(numpy_typing, 'NBitBase', NBitBase)
setattr(numpy_typing, 'NDArray', NDArray)

if 'numpy.typing' not in sys.modules:
    sys.modules['numpy.typing'] = types.ModuleType('numpy.typing')
numpy_typing_public = sys.modules['numpy.typing']
setattr(numpy_typing_public, 'ArrayLike', ArrayLike)
setattr(numpy_typing_public, 'DTypeLike', DTypeLike)
setattr(numpy_typing_public, 'NBitBase', NBitBase)
setattr(numpy_typing_public, 'NDArray', NDArray)

# ====================== 配置参数 ======================
def parse_args():
    """解析命令行参数"""
    parser = argparse.ArgumentParser(description="BERT + CNN 情感分类器")
    parser.add_argument('--mode', choices=['train', 'test'], required=True, help='运行模式: train or test')
    parser.add_argument('--model_path', type=str, default='./model/cnn_bert_final.model', help='模型保存/加载路径')
    parser.add_argument('--epochs', type=int, default=5, help='训练轮数')
    parser.add_argument('--batch_size', type=int, default=24, help='批处理大小')
    parser.add_argument('--lr', type=float, default=5e-4, help='学习率')
    parser.add_argument('--num_filters', type=int, default=128, help='卷积核数量')
    parser.add_argument('--filter_sizes', type=str, default="2,3,4", help='卷积核大小，逗号分隔')
    parser.add_argument('--max_length', type=int, default=128, help='文本最大长度')
    parser.add_argument('--bert_path', type=str, default=r'G:\中文情感识别\bert_base_uncased', help='BERT模型本地路径')
    parser.add_argument('--data_dir', type=str, default=r'G:\中文情感识别\data\ChnSentiCorp_htl_all', help='数据目录')
    parser.add_argument('--freeze_bert', type=int, default=1, choices=[0, 1], help='是否冻结BERT参数: 1(冻结), 0(不冻结)')
    parser.add_argument('--dropout', type=float, default=0.5, help='Dropout概率')
    parser.add_argument('--log_interval', type=int, default=10, help='日志打印间隔')
    
    # 检查是否在Jupyter环境中运行
    is_jupyter = 'ipykernel' in sys.modules
    if is_jupyter:
        sys.argv = [sys.argv[0], '--mode', 'train']
    
    return parser.parse_args([]) if is_jupyter and len(sys.argv) < 2 else parser.parse_args()

# ====================== 数据加载 ======================
def load_corpus_bert(path):
    """加载CSV格式的数据集"""
    try:
        # 尝试不同编码
        for encoding in ['utf-8', 'gbk', 'latin1']:
            try:
                df = pd.read_csv(path, encoding=encoding)
                print(f"使用 {encoding} 编码成功读取文件: {path}")
                
                # 检查列名是否包含text/label
                text_col, label_col = None, None
                for col in df.columns:
                    col_lower = str(col).lower()
                    if 'text' in col_lower or '内容' in col_lower or '评论' in col_lower or 'review' in col_lower:
                        text_col = col
                    elif 'label' in col_lower or '标签' in col_lower or 'sentiment' in col_lower:
                        label_col = col
                
                # 如果找到合适的列名
                if text_col and label_col:
                    print(f"检测到列名: text={text_col}, label={label_col}")
                    return [(row[text_col], row[label_col]) for _, row in df.iterrows()]
                
                # 如果没有标准列头，但有两列数据
                if len(df.columns) == 2:
                    print(f"检测到无列头文件，使用第一列为标签，第二列为文本: {path}")
                    return [(row[1], row[0]) for _, row in df.iterrows()]
                
                # 如果只有一列数据，尝试解析
                if len(df.columns) == 1:
                    print(f"检测到单列文件，尝试解析标签和文本: {path}")
                    data = []
                    for _, row in df.iterrows():
                        parts = str(row[0]).split('\t')
                        if len(parts) == 2:
                            data.append((parts[1], float(parts[0])))
                    if data:
                        print(f"成功解析 {len(data)} 条记录")
                        return data
                    
                # 如果以上都不符合
                print(f"无法识别文件格式，列名: {df.columns.tolist()}")
                return []
            except UnicodeDecodeError:
                print(f"编码 {encoding} 失败，尝试下一种编码...")
                continue
            except Exception as e:
                print(f"读取文件时出错: {e}")
                return []
    except Exception as e:
        print(f"加载数据失败: {e}")
        return []

# ====================== 数据集类 ======================
class SentimentDataset(Dataset):
    """情感分析数据集类"""
    def __init__(self, data, tokenizer, max_length=128):
        self.data = []
        self.labels = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        print(f"正在处理 {len(data)} 个样本...")
        
        for i, (text, label) in enumerate(data):
            try:
                # 确保文本是字符串
                if not isinstance(text, str):
                    text = str(text)
                
                # 处理空文本
                if not text.strip():
                    text = "[空文本]"
                    
                inputs = tokenizer(
                    text, 
                    padding='max_length', 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=max_length
                )
                self.data.append(inputs)
                self.labels.append(float(label))
                
                # 显示进度
                if (i + 1) % 100 == 0:
                    print(f"已处理 {i+1}/{len(data)} 个样本")
            except Exception as e:
                print(f"处理文本失败: {e}")
                print(f"问题文本: {text[:50] if isinstance(text, str) else '非字符串类型'}...")
                self.data.append(tokenizer(
                    "[处理失败]", 
                    padding='max_length', 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=max_length
                ))
                self.labels.append(0.0)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]

    def __len__(self):
        return len(self.labels)

def collate_fn(data):
    """批处理函数"""
    try:
        input_ids = [item[0]['input_ids'].squeeze(0) for item in data]
        attention_mask = [item[0]['attention_mask'].squeeze(0) for item in data]
        labels = [item[1] for item in data]
        
        input_ids = torch.stack(input_ids)
        attention_mask = torch.stack(attention_mask)
        return input_ids, attention_mask, torch.tensor(labels)
    except Exception as e:
        print(f"数据批处理失败: {e}")
        # 打印问题数据
        for i, item in enumerate(data):
            print(f"样本 {i}:")
            print(f"输入ID形状: {item[0]['input_ids'].shape}")
            print(f"注意力掩码形状: {item[0]['attention_mask'].shape}")
            print(f"标签: {item[1]}")
        raise

# ====================== BERT-CNN 模型 ======================
class BERT_CNN(nn.Module):
    """BERT + CNN 情感分类模型"""
    def __init__(self, bert_model, filter_sizes, num_filters, dropout=0.5, freeze_bert=True):
        super(BERT_CNN, self).__init__()
        self.bert = bert_model
        
        # 冻结BERT参数
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
            print("冻结BERT参数")
        else:
            print("微调BERT参数")

        # 卷积层
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, 768)) for k in filter_sizes
        ])
        
        # 分类层
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        # BERT编码
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = bert_output.last_hidden_state

        # 添加通道维度
        x = x.unsqueeze(1) 
        
        # 多尺寸卷积
        conv_outputs = []
        for conv in self.convs:
            conv_out = torch.relu(conv(x)).squeeze(3) 
            pooled = torch.max(conv_out, dim=2)[0]    
            conv_outputs.append(pooled)
            
        # 拼接特征
        x = torch.cat(conv_outputs, dim=1) 
        x = self.dropout(x)
        logits = self.fc(x)
        return self.sigmoid(logits)

# ====================== 训练和评估函数 ======================
def evaluate(model, data_loader, device):
    """评估模型性能"""
    model.eval()
    y_pred, y_true = [], []
    total_loss = 0.0
    criterion = nn.BCELoss()
    
    start_time = time.time()
    with torch.no_grad():
        for i, (input_ids, attention_mask, labels) in enumerate(data_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()
            
            outputs = model(input_ids, attention_mask).view(-1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            y_pred.append(outputs.cpu())
            y_true.append(labels.cpu())
            
            # 显示进度
            if (i + 1) % 10 == 0:
                print(f"已评估 {i+1}/{len(data_loader)} 批次")

    # 计算指标
    y_prob = torch.cat(y_pred).numpy()
    y_true = torch.cat(y_true).numpy()
    y_label = (y_prob > 0.5).astype(int)
    
    accuracy = metrics.accuracy_score(y_true, y_label)
    precision = metrics.precision_score(y_true, y_label)
    recall = metrics.recall_score(y_true, y_label)
    f1 = metrics.f1_score(y_true, y_label)
    auc = metrics.roc_auc_score(y_true, y_prob)
    avg_loss = total_loss / len(data_loader)
    
    # 打印结果
    print("\n分类报告:")
    print(metrics.classification_report(y_true, y_label))
    print(f"评估时间: {time.time()-start_time:.2f}秒")
    print(f"损失: {avg_loss:.4f}, 准确率: {accuracy:.4f}, AUC: {auc:.4f}")
    print(f"精确率: {precision:.4f}, 召回率: {recall:.4f}, F1分数: {f1:.4f}")
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

def train_model(model, train_loader, dev_loader, optimizer, criterion, args, device):
    """训练模型"""
    best_f1 = 0.0
    history = {'train_loss': [], 'dev_metrics': []}
    
    # 创建模型目录
    os.makedirs(os.path.dirname(args.model_path), exist_ok=True)
    
    print(f"\n开始训练，共 {args.epochs} 个周期，批量大小: {args.batch_size}，学习率: {args.lr}")
    print(f"模型结构: {model}")
    
    for epoch in range(args.epochs):
        model.train()
        total_loss, total_samples = 0, 0
        epoch_start = time.time()
        
        print(f"\n{'='*50}")
        print(f"周期 {epoch + 1}/{args.epochs}")
        
        for i, (input_ids, attention_mask, labels) in enumerate(train_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()

            # 前向传播
            outputs = model(input_ids, attention_mask).view(-1)
            loss = criterion(outputs, labels)
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 统计指标
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            
            # 打印日志
            if (i + 1) % args.log_interval == 0:
                avg_loss = total_loss / total_samples
                print(f"批次 {i + 1}/{len(train_loader)} - 损失: {avg_loss:.4f}")
                total_loss, total_samples = 0, 0
        
        # 计算训练损失
        train_loss = total_loss / total_samples if total_samples > 0 else 0.0
        history['train_loss'].append(train_loss)
        
        # 验证集评估
        print(f"\n周期 {epoch+1}/{args.epochs} 验证结果:")
        dev_metrics = evaluate(model, dev_loader, device)
        history['dev_metrics'].append(dev_metrics)
        
        # 保存最佳模型
        if dev_metrics['f1'] > best_f1:
            best_f1 = dev_metrics['f1']
            torch.save(model.state_dict(), args.model_path)
            print(f"新最佳模型已保存至 {args.model_path}, F1={best_f1:.4f}")
        
        # 打印周期摘要
        epoch_time = time.time() - epoch_start
        print(f"周期 {epoch+1} 完成, 耗时: {epoch_time//60:.0f}分 {epoch_time%60:.0f}秒")
        print(f"训练损失: {train_loss:.4f}, 验证F1: {dev_metrics['f1']:.4f}")
    
    print(f"\n训练完成，最佳验证F1: {best_f1:.4f}")
    return history

# ====================== 主函数 ======================
def main():
    # 解析参数
    args = parse_args()
    
    # 解析卷积核大小
    filter_sizes = list(map(int, args.filter_sizes.split(',')))
    
    # 设置设备
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"\n{'='*50}")
    print(f"运行模式: {args.mode}")
    print(f"设备: {device}")
    print(f"BERT路径: {args.bert_path}")
    print(f"数据目录: {args.data_dir}")
    print(f"模型路径: {args.model_path}")
    print(f"训练轮数: {args.epochs}, 批量大小: {args.batch_size}, 学习率: {args.lr}")
    print(f"卷积核大小: {filter_sizes}, 卷积核数量: {args.num_filters}")
    print(f"最大长度: {args.max_length}, Dropout: {args.dropout}")
    print(f"冻结BERT: {'是' if args.freeze_bert else '否'}")
    print(f"{'='*50}\n")
    
    # 加载BERT模型
    print(f"从本地路径加载BERT模型: {args.bert_path}")
    if not os.path.exists(args.bert_path):
        raise FileNotFoundError(f"BERT本地路径不存在: {args.bert_path}")
    
    tokenizer = BertTokenizer.from_pretrained(args.bert_path)
    bert_model = BertModel.from_pretrained(args.bert_path).to(device)
    
    # 加载数据
    print("\n加载数据集...")
    train_path = os.path.join(args.data_dir, 'train.csv')
    dev_path = os.path.join(args.data_dir, 'dev.csv')
    test_path = os.path.join(args.data_dir, 'test.csv')
    
    print(f"训练数据: {train_path}")
    train_data = load_corpus_bert(train_path)
    print(f"验证数据: {dev_path}")
    dev_data = load_corpus_bert(dev_path)
    print(f"测试数据: {test_path}")
    test_data = load_corpus_bert(test_path)
    
    # 检查数据
    if not train_data:
        raise ValueError("训练数据为空，请检查数据文件")
    if not dev_data:
        print("警告: 验证数据为空，将使用部分训练数据作为验证集")
        dev_data = train_data[:len(train_data)//5]
        train_data = train_data[len(train_data)//5:]
    if not test_data:
        print("警告: 测试数据为空，将使用验证数据作为测试集")
        test_data = dev_data
    
    print(f"\n训练样本数: {len(train_data)}")
    print(f"验证样本数: {len(dev_data)}")
    print(f"测试样本数: {len(test_data)}")
    
    # 创建数据加载器
    print("\n创建数据加载器...")
    train_dataset = SentimentDataset(train_data, tokenizer, args.max_length)
    dev_dataset = SentimentDataset(dev_data, tokenizer, args.max_length)
    test_dataset = SentimentDataset(test_data, tokenizer, args.max_length)
    
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, 
                             collate_fn=collate_fn, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=args.batch_size, 
                           collate_fn=collate_fn, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, 
                            collate_fn=collate_fn, shuffle=False)
    
    # 创建模型
    model = BERT_CNN(
        bert_model, 
        filter_sizes=filter_sizes,
        num_filters=args.num_filters,
        dropout=args.dropout,
        freeze_bert=args.freeze_bert
    ).to(device)
    
    # 根据模式运行
    if args.mode == 'train':
        # 训练模型
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        
        # 学习率调度器
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=1, verbose=True
        )
        
        print("\n开始训练模型...")
        history = train_model(
            model, train_loader, dev_loader, optimizer, criterion, args, device
        )
        
        # 最终测试
        print("\n最终测试结果:")
        test_metrics = evaluate(model, test_loader, device)
        
    elif args.mode == 'test':
        # 测试模型
        model.load_state_dict(torch.load(args.model_path, map_location=device))
        print("模型加载成功，测试结果:")
        test_metrics = evaluate(model, test_loader, device)
    
    print("\n程序执行完成")

if __name__ == '__main__':
    main()


运行模式: train
设备: cpu
BERT路径: G:\中文情感识别\bert_base_uncased
数据目录: G:\中文情感识别\data\ChnSentiCorp_htl_all
模型路径: ./model/cnn_bert_final.model
训练轮数: 5, 批量大小: 24, 学习率: 0.0005
卷积核大小: [2, 3, 4], 卷积核数量: 128
最大长度: 128, Dropout: 0.5
冻结BERT: 是

从本地路径加载BERT模型: G:\中文情感识别\bert_base_uncased

加载数据集...
训练数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
验证数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
测试数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv

训练样本数: 4981
验证样本数: 1884
测试样本数: 899

创建数据加载器...
正在处理 4981 个样本...
已处理 100/4981 个样本
已处理 200/4981 个样本
已处理 300/4981 个样本
已处理 400/4981 个样本
已处理 500/4981 个样本
已处理 600/49

TypeError: __init__() got an unexpected keyword argument 'verbose'

In [26]:
import numpy as np
import sys
import types
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn import metrics
import pandas as pd
import argparse
import os
import time
import torch
from transformers import BertTokenizer, BertModel

# ====================== 解决NumPy导入问题 ======================
if 'numpy._typing' not in sys.modules:
    sys.modules['numpy._typing'] = types.ModuleType('numpy._typing')

class _NBitLong: pass
class ArrayLike: pass
class DTypeLike: pass
class NBitBase: pass
class NDArray: pass

numpy_typing = sys.modules['numpy._typing']
setattr(numpy_typing, '_NBitLong', _NBitLong)
setattr(numpy_typing, 'ArrayLike', ArrayLike)
setattr(numpy_typing, 'DTypeLike', DTypeLike)
setattr(numpy_typing, 'NBitBase', NBitBase)
setattr(numpy_typing, 'NDArray', NDArray)

if 'numpy.typing' not in sys.modules:
    sys.modules['numpy.typing'] = types.ModuleType('numpy.typing')
numpy_typing_public = sys.modules['numpy.typing']
setattr(numpy_typing_public, 'ArrayLike', ArrayLike)
setattr(numpy_typing_public, 'DTypeLike', DTypeLike)
setattr(numpy_typing_public, 'NBitBase', NBitBase)
setattr(numpy_typing_public, 'NDArray', NDArray)

# ====================== 配置参数 ======================
def parse_args():
    """解析命令行参数"""
    parser = argparse.ArgumentParser(description="BERT + CNN 情感分类器")
    parser.add_argument('--mode', choices=['train', 'test'], required=True, help='运行模式: train or test')
    parser.add_argument('--model_path', type=str, default='./model/cnn_bert_final.model', help='模型保存/加载路径')
    parser.add_argument('--epochs', type=int, default=5, help='训练轮数')
    parser.add_argument('--batch_size', type=int, default=24, help='批处理大小')
    parser.add_argument('--lr', type=float, default=5e-4, help='学习率')
    parser.add_argument('--num_filters', type=int, default=128, help='卷积核数量')
    parser.add_argument('--filter_sizes', type=str, default="2,3,4", help='卷积核大小，逗号分隔')
    parser.add_argument('--max_length', type=int, default=128, help='文本最大长度')
    parser.add_argument('--bert_path', type=str, default=r'G:\中文情感识别\bert_base_uncased', help='BERT模型本地路径')
    parser.add_argument('--data_dir', type=str, default=r'G:\中文情感识别\data\ChnSentiCorp_htl_all', help='数据目录')
    parser.add_argument('--freeze_bert', type=int, default=1, choices=[0, 1], help='是否冻结BERT参数: 1(冻结), 0(不冻结)')
    parser.add_argument('--dropout', type=float, default=0.5, help='Dropout概率')
    parser.add_argument('--log_interval', type=int, default=10, help='日志打印间隔')
    
    # 检查是否在Jupyter环境中运行
    is_jupyter = 'ipykernel' in sys.modules
    if is_jupyter:
        sys.argv = [sys.argv[0], '--mode', 'train']
    
    return parser.parse_args([]) if is_jupyter and len(sys.argv) < 2 else parser.parse_args()

# ====================== 数据加载 ======================
def load_corpus_bert(path):
    """加载CSV格式的数据集"""
    try:
        # 尝试不同编码
        for encoding in ['utf-8', 'gbk', 'latin1']:
            try:
                df = pd.read_csv(path, encoding=encoding)
                print(f"使用 {encoding} 编码成功读取文件: {path}")
                
                # 检查列名是否包含text/label
                text_col, label_col = None, None
                for col in df.columns:
                    col_lower = str(col).lower()
                    if 'text' in col_lower or '内容' in col_lower or '评论' in col_lower or 'review' in col_lower:
                        text_col = col
                    elif 'label' in col_lower or '标签' in col_lower or 'sentiment' in col_lower:
                        label_col = col
                
                # 如果找到合适的列名
                if text_col and label_col:
                    print(f"检测到列名: text={text_col}, label={label_col}")
                    return [(row[text_col], row[label_col]) for _, row in df.iterrows()]
                
                # 如果没有标准列头，但有两列数据
                if len(df.columns) == 2:
                    print(f"检测到无列头文件，使用第一列为标签，第二列为文本: {path}")
                    return [(row[1], row[0]) for _, row in df.iterrows()]
                
                # 如果只有一列数据，尝试解析
                if len(df.columns) == 1:
                    print(f"检测到单列文件，尝试解析标签和文本: {path}")
                    data = []
                    for _, row in df.iterrows():
                        parts = str(row[0]).split('\t')
                        if len(parts) == 2:
                            data.append((parts[1], float(parts[0])))
                    if data:
                        print(f"成功解析 {len(data)} 条记录")
                        return data
                    
                # 如果以上都不符合
                print(f"无法识别文件格式，列名: {df.columns.tolist()}")
                return []
            except UnicodeDecodeError:
                print(f"编码 {encoding} 失败，尝试下一种编码...")
                continue
            except Exception as e:
                print(f"读取文件时出错: {e}")
                return []
    except Exception as e:
        print(f"加载数据失败: {e}")
        return []

# ====================== 数据集类 ======================
class SentimentDataset(Dataset):
    """情感分析数据集类"""
    def __init__(self, data, tokenizer, max_length=128):
        self.data = []
        self.labels = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        print(f"正在处理 {len(data)} 个样本...")
        
        for i, (text, label) in enumerate(data):
            try:
                # 确保文本是字符串
                if not isinstance(text, str):
                    text = str(text)
                
                # 处理空文本
                if not text.strip():
                    text = "[空文本]"
                    
                inputs = tokenizer(
                    text, 
                    padding='max_length', 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=max_length
                )
                self.data.append(inputs)
                self.labels.append(float(label))
                
                # 显示进度
                if (i + 1) % 100 == 0:
                    print(f"已处理 {i+1}/{len(data)} 个样本")
            except Exception as e:
                print(f"处理文本失败: {e}")
                print(f"问题文本: {text[:50] if isinstance(text, str) else '非字符串类型'}...")
                self.data.append(tokenizer(
                    "[处理失败]", 
                    padding='max_length', 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=max_length
                ))
                self.labels.append(0.0)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]

    def __len__(self):
        return len(self.labels)

def collate_fn(data):
    """批处理函数"""
    try:
        input_ids = [item[0]['input_ids'].squeeze(0) for item in data]
        attention_mask = [item[0]['attention_mask'].squeeze(0) for item in data]
        labels = [item[1] for item in data]
        
        input_ids = torch.stack(input_ids)
        attention_mask = torch.stack(attention_mask)
        return input_ids, attention_mask, torch.tensor(labels)
    except Exception as e:
        print(f"数据批处理失败: {e}")
        # 打印问题数据
        for i, item in enumerate(data):
            print(f"样本 {i}:")
            print(f"输入ID形状: {item[0]['input_ids'].shape}")
            print(f"注意力掩码形状: {item[0]['attention_mask'].shape}")
            print(f"标签: {item[1]}")
        raise

# ====================== BERT-CNN 模型 ======================
class BERT_CNN(nn.Module):
    """BERT + CNN 情感分类模型"""
    def __init__(self, bert_model, filter_sizes, num_filters, dropout=0.5, freeze_bert=True):
        super(BERT_CNN, self).__init__()
        self.bert = bert_model
        
        # 冻结BERT参数
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
            print("冻结BERT参数")
        else:
            print("微调BERT参数")

        # 卷积层
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, 768)) for k in filter_sizes
        ])
        
        # 分类层
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        # BERT编码
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = bert_output.last_hidden_state

        # 添加通道维度
        x = x.unsqueeze(1) 
        
        # 多尺寸卷积
        conv_outputs = []
        for conv in self.convs:
            conv_out = torch.relu(conv(x)).squeeze(3) 
            pooled = torch.max(conv_out, dim=2)[0]    
            conv_outputs.append(pooled)
            
        # 拼接特征
        x = torch.cat(conv_outputs, dim=1) 
        x = self.dropout(x)
        logits = self.fc(x)
        return self.sigmoid(logits)

# ====================== 训练和评估函数 ======================
def evaluate(model, data_loader, device):
    """评估模型性能"""
    model.eval()
    y_pred, y_true = [], []
    total_loss = 0.0
    criterion = nn.BCELoss()
    
    start_time = time.time()
    with torch.no_grad():
        for i, (input_ids, attention_mask, labels) in enumerate(data_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()
            
            outputs = model(input_ids, attention_mask).view(-1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            y_pred.append(outputs.cpu())
            y_true.append(labels.cpu())
            
            # 显示进度
            if (i + 1) % 10 == 0:
                print(f"已评估 {i+1}/{len(data_loader)} 批次")

    # 计算指标
    y_prob = torch.cat(y_pred).numpy()
    y_true = torch.cat(y_true).numpy()
    y_label = (y_prob > 0.5).astype(int)
    
    accuracy = metrics.accuracy_score(y_true, y_label)
    precision = metrics.precision_score(y_true, y_label)
    recall = metrics.recall_score(y_true, y_label)
    f1 = metrics.f1_score(y_true, y_label)
    auc = metrics.roc_auc_score(y_true, y_prob)
    avg_loss = total_loss / len(data_loader)
    
    # 打印结果
    print("\n分类报告:")
    print(metrics.classification_report(y_true, y_label))
    print(f"评估时间: {time.time()-start_time:.2f}秒")
    print(f"损失: {avg_loss:.4f}, 准确率: {accuracy:.4f}, AUC: {auc:.4f}")
    print(f"精确率: {precision:.4f}, 召回率: {recall:.4f}, F1分数: {f1:.4f}")
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

def train_model(model, train_loader, dev_loader, optimizer, criterion, args, device):
    """训练模型"""
    best_f1 = 0.0
    history = {'train_loss': [], 'dev_metrics': []}
    
    # 创建模型目录
    os.makedirs(os.path.dirname(args.model_path), exist_ok=True)
    
    print(f"\n开始训练，共 {args.epochs} 个周期，批量大小: {args.batch_size}，学习率: {args.lr}")
    print(f"模型结构: {model}")
    
    for epoch in range(args.epochs):
        model.train()
        total_loss, total_samples = 0, 0
        epoch_start = time.time()
        
        print(f"\n{'='*50}")
        print(f"周期 {epoch + 1}/{args.epochs}")
        
        for i, (input_ids, attention_mask, labels) in enumerate(train_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device).float()

            # 前向传播
            outputs = model(input_ids, attention_mask).view(-1)
            loss = criterion(outputs, labels)
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 统计指标
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            
            # 打印日志
            if (i + 1) % args.log_interval == 0:
                avg_loss = total_loss / total_samples
                print(f"批次 {i + 1}/{len(train_loader)} - 损失: {avg_loss:.4f}")
                total_loss, total_samples = 0, 0
        
        # 计算训练损失
        train_loss = total_loss / total_samples if total_samples > 0 else 0.0
        history['train_loss'].append(train_loss)
        
        # 验证集评估
        print(f"\n周期 {epoch+1}/{args.epochs} 验证结果:")
        dev_metrics = evaluate(model, dev_loader, device)
        history['dev_metrics'].append(dev_metrics)
        
        # 保存最佳模型
        if dev_metrics['f1'] > best_f1:
            best_f1 = dev_metrics['f1']
            torch.save(model.state_dict(), args.model_path)
            print(f"新最佳模型已保存至 {args.model_path}, F1={best_f1:.4f}")
        
        # 打印周期摘要
        epoch_time = time.time() - epoch_start
        print(f"周期 {epoch+1} 完成, 耗时: {epoch_time//60:.0f}分 {epoch_time%60:.0f}秒")
        print(f"训练损失: {train_loss:.4f}, 验证F1: {dev_metrics['f1']:.4f}")
    
    print(f"\n训练完成，最佳验证F1: {best_f1:.4f}")
    return history

# ====================== 主函数 ======================
def main():
    # 解析参数
    args = parse_args()
    
    # 解析卷积核大小
    filter_sizes = list(map(int, args.filter_sizes.split(',')))
    
    # 设置设备
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"\n{'='*50}")
    print(f"运行模式: {args.mode}")
    print(f"设备: {device}")
    print(f"BERT路径: {args.bert_path}")
    print(f"数据目录: {args.data_dir}")
    print(f"模型路径: {args.model_path}")
    print(f"训练轮数: {args.epochs}, 批量大小: {args.batch_size}, 学习率: {args.lr}")
    print(f"卷积核大小: {filter_sizes}, 卷积核数量: {args.num_filters}")
    print(f"最大长度: {args.max_length}, Dropout: {args.dropout}")
    print(f"冻结BERT: {'是' if args.freeze_bert else '否'}")
    print(f"{'='*50}\n")
    
    # 加载BERT模型
    print(f"从本地路径加载BERT模型: {args.bert_path}")
    if not os.path.exists(args.bert_path):
        raise FileNotFoundError(f"BERT本地路径不存在: {args.bert_path}")
    
    tokenizer = BertTokenizer.from_pretrained(args.bert_path)
    bert_model = BertModel.from_pretrained(args.bert_path).to(device)
    
    # 加载数据
    print("\n加载数据集...")
    train_path = os.path.join(args.data_dir, 'train.csv')
    dev_path = os.path.join(args.data_dir, 'dev.csv')
    test_path = os.path.join(args.data_dir, 'test.csv')
    
    print(f"训练数据: {train_path}")
    train_data = load_corpus_bert(train_path)
    print(f"验证数据: {dev_path}")
    dev_data = load_corpus_bert(dev_path)
    print(f"测试数据: {test_path}")
    test_data = load_corpus_bert(test_path)
    
    # 检查数据
    if not train_data:
        raise ValueError("训练数据为空，请检查数据文件")
    if not dev_data:
        print("警告: 验证数据为空，将使用部分训练数据作为验证集")
        dev_data = train_data[:len(train_data)//5]
        train_data = train_data[len(train_data)//5:]
    if not test_data:
        print("警告: 测试数据为空，将使用验证数据作为测试集")
        test_data = dev_data
    
    print(f"\n训练样本数: {len(train_data)}")
    print(f"验证样本数: {len(dev_data)}")
    print(f"测试样本数: {len(test_data)}")
    
    # 创建数据加载器
    print("\n创建数据加载器...")
    train_dataset = SentimentDataset(train_data, tokenizer, args.max_length)
    dev_dataset = SentimentDataset(dev_data, tokenizer, args.max_length)
    test_dataset = SentimentDataset(test_data, tokenizer, args.max_length)
    
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, 
                             collate_fn=collate_fn, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=args.batch_size, 
                           collate_fn=collate_fn, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, 
                            collate_fn=collate_fn, shuffle=False)
    
    # 创建模型
    model = BERT_CNN(
        bert_model, 
        filter_sizes=filter_sizes,
        num_filters=args.num_filters,
        dropout=args.dropout,
        freeze_bert=args.freeze_bert
    ).to(device)
    
    # 根据模式运行
    if args.mode == 'train':
        # 训练模型
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        
        # 学习率调度器 - 移除了verbose参数
        # 如果需要，可以添加更简单的调度器
        # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)
        
        print("\n开始训练模型...")
        history = train_model(
            model, train_loader, dev_loader, optimizer, criterion, args, device
        )
        
        # 最终测试
        print("\n最终测试结果:")
        test_metrics = evaluate(model, test_loader, device)
        
    elif args.mode == 'test':
        # 测试模型
        model.load_state_dict(torch.load(args.model_path, map_location=device))
        print("模型加载成功，测试结果:")
        test_metrics = evaluate(model, test_loader, device)
    
    print("\n程序执行完成")

if __name__ == '__main__':
    main()


运行模式: train
设备: cpu
BERT路径: G:\中文情感识别\bert_base_uncased
数据目录: G:\中文情感识别\data\ChnSentiCorp_htl_all
模型路径: ./model/cnn_bert_final.model
训练轮数: 5, 批量大小: 24, 学习率: 0.0005
卷积核大小: [2, 3, 4], 卷积核数量: 128
最大长度: 128, Dropout: 0.5
冻结BERT: 是

从本地路径加载BERT模型: G:\中文情感识别\bert_base_uncased

加载数据集...
训练数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\train.csv
验证数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\dev.csv
测试数据: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv
使用 utf-8 编码成功读取文件: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv
检测到无列头文件，使用第一列为标签，第二列为文本: G:\中文情感识别\data\ChnSentiCorp_htl_all\test.csv

训练样本数: 4981
验证样本数: 1884
测试样本数: 899

创建数据加载器...
正在处理 4981 个样本...
已处理 100/4981 个样本
已处理 200/4981 个样本
已处理 300/4981 个样本
已处理 400/4981 个样本
已处理 500/4981 个样本
已处理 600/49

批次 110/208 - 损失: 0.1737
批次 120/208 - 损失: 0.1778
批次 130/208 - 损失: 0.1578
批次 140/208 - 损失: 0.1976
批次 150/208 - 损失: 0.1712
批次 160/208 - 损失: 0.1355
批次 170/208 - 损失: 0.1947
批次 180/208 - 损失: 0.1883
批次 190/208 - 损失: 0.1880
批次 200/208 - 损失: 0.1909

周期 4/5 验证结果:
已评估 10/79 批次
已评估 20/79 批次
已评估 30/79 批次
已评估 40/79 批次
已评估 50/79 批次
已评估 60/79 批次
已评估 70/79 批次

分类报告:
              precision    recall  f1-score   support

         0.0       0.90      0.85      0.87       837
         1.0       0.88      0.92      0.90      1047

    accuracy                           0.89      1884
   macro avg       0.89      0.89      0.89      1884
weighted avg       0.89      0.89      0.89      1884

评估时间: 141.76秒
损失: 0.2542, 准确率: 0.8901, AUC: 0.9634
精确率: 0.8832, 召回率: 0.9245, F1分数: 0.9034
周期 4 完成, 耗时: 13分 53秒
训练损失: 0.1699, 验证F1: 0.9034

周期 5/5
批次 10/208 - 损失: 0.1453
批次 20/208 - 损失: 0.1483
批次 30/208 - 损失: 0.1352
批次 40/208 - 损失: 0.1408
批次 50/208 - 损失: 0.1165
批次 60/208 - 损失: 0.0943
批次 70/208 - 损失: 0.1155
批次 80/208 - 损失